# Chapter 6: Spark Structured Streaming — Real-Time Flight Delay Monitor

**Scenario**: Simulate a real-time flight status feed using Kafka.
- **Source**: Kafka topic `flight-stream` (fed by `kafka_producer.py`)
- **Processing**: Windowed aggregation — average delay per carrier in a 1-minute tumbling window
- **Alert**: Flag any carrier with average delay > 45 minutes in a window
- **Sink**: Console output (live updating)

## 1. Setup SparkSession with Kafka support

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

spark = SparkSession.builder \
    .appName("KafkaStructuredStreaming") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession created successfully.")

SparkSession created successfully.


## 2. Define schema & read from Kafka

In [2]:
# Schema khớp với record gửi từ kafka_producer.py
flight_schema = StructType([
    StructField("AIRLINE",         StringType(),  True),
    StructField("ORIGIN_AIRPORT",  StringType(),  True),
    StructField("DEST_AIRPORT",    StringType(),  True),
    StructField("DEPARTURE_DELAY", FloatType(),   True),
    StructField("ARRIVAL_DELAY",   FloatType(),   True),
    StructField("DISTANCE",        FloatType(),   True),
    StructField("CANCELLED",       IntegerType(), True),
])

# Read stream từ Kafka
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "flight-stream") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print("Kafka stream connected.")

Kafka stream connected.


## 3. Parse JSON records

In [3]:
# Kafka gửi value dạng bytes → parse JSON → extract fields
parsed_stream = raw_stream \
    .select(
        F.from_json(
            F.col("value").cast("string"),
            flight_schema
        ).alias("data"),
        F.col("timestamp").alias("event_time")
    ) \
    .select("data.*", "event_time") \
    .filter(F.col("CANCELLED") == 0) \
    .filter(F.col("ARRIVAL_DELAY").isNotNull())

print("Stream schema:")
parsed_stream.printSchema()

Stream schema:
root
 |-- AIRLINE: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DEST_AIRPORT: string (nullable = true)
 |-- DEPARTURE_DELAY: float (nullable = true)
 |-- ARRIVAL_DELAY: float (nullable = true)
 |-- DISTANCE: float (nullable = true)
 |-- CANCELLED: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)



## 4. Windowed Aggregation — Average delay per carrier (1-minute tumbling window)

In [4]:
windowed_stats = parsed_stream \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(
        F.window(F.col("event_time"), "1 minute"),
        F.col("AIRLINE")
    ) \
    .agg(
        F.avg("ARRIVAL_DELAY").alias("avg_arrival_delay"),
        F.avg("DEPARTURE_DELAY").alias("avg_departure_delay"),
        F.count("*").alias("flight_count"),
        F.sum(F.when(F.col("ARRIVAL_DELAY") > 15, 1).otherwise(0)).alias("delayed_flights")
    ) \
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("AIRLINE"),
        F.col("avg_arrival_delay"),
        F.col("avg_departure_delay"),
        F.col("flight_count"),
        F.col("delayed_flights"),
        F.round(F.col("delayed_flights") / F.col("flight_count") * 100, 2).alias("delay_rate_pct")
    )

print("Windowed aggregation defined.")

Windowed aggregation defined.


## 5. Alert Logic — Flag carriers with avg delay > 45 minutes

In [5]:
ALERT_THRESHOLD = 15.0   # minutes

alerting_stream = windowed_stats \
    .withColumn(
        "ALERT",
        F.when(
            F.col("avg_arrival_delay") > ALERT_THRESHOLD,
            F.lit(f"WARNING: HIGH DELAY (>{ALERT_THRESHOLD} min)")
        ).otherwise(F.lit("OK: Normal"))
    ) \
    .orderBy(F.col("avg_arrival_delay").desc())

print(f"Alert threshold: avg arrival delay > {ALERT_THRESHOLD} minutes")

Alert threshold: avg arrival delay > 15.0 minutes


## 6. Output to console (live updating)

In [10]:
# Query 1: console (giữ để xem live)
query_console = alerting_stream \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 50) \
    .trigger(processingTime="30 seconds") \
    .start()

# Query 2: memory sink (để lấy bảng kết quả cho báo cáo)
query_memory = alerting_stream \
    .writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("flight_alerts") \
    .trigger(processingTime="30 seconds") \
    .start()

query = query_console  # giữ tên query cho cell monitoring bên dưới

print("=" * 60)
print("  Streaming started! Waiting for data from Kafka...")
print(f"  Alert threshold: avg delay > {ALERT_THRESHOLD} min")
print("  Processing window: 1 minute tumbling")
print("  Trigger interval: every 30 seconds")
print("=" * 60)

  Streaming started! Waiting for data from Kafka...
  Alert threshold: avg delay > 15.0 min
  Processing window: 1 minute tumbling
  Trigger interval: every 30 seconds


## 7. Monitor stream status

In [11]:
import time

DEMO_DURATION = 600

print(f"Running for {DEMO_DURATION // 60} minutes...")
start = time.time()

while query.isActive:
    status = query.status
    progress = query.lastProgress

    print(f"\n[{time.strftime('%H:%M:%S')}] Stream status: {status['message']}")

    if progress:
        print(f"  Input rows/sec:     {progress.get('inputRowsPerSecond', 0):.1f}")
        print(f"  Processed rows/sec: {progress.get('processedRowsPerSecond', 0):.1f}")
        print(f"  Batch ID:           {progress.get('batchId', 'N/A')}")

        # In bảng kết quả sau mỗi batch — đây là phần lấy minh chứng
        print("\n--- Kết quả batch hiện tại ---")
        spark.sql("""
            SELECT * FROM flight_alerts
            ORDER BY avg_arrival_delay DESC
        """).show(20, truncate=False)

    if time.time() - start > DEMO_DURATION:
        print("\nDemo duration reached. Stopping stream...")
        query.stop()
        query_memory.stop()
        break

    time.sleep(30)

Running for 10 minutes...

[23:12:08] Stream status: Processing new data

[23:12:38] Stream status: Processing new data

[23:13:08] Stream status: Processing new data

[23:13:38] Stream status: Processing new data
  Input rows/sec:     0.0
  Processed rows/sec: 0.4
  Batch ID:           0

--- Kết quả batch hiện tại ---
+------------+----------+-------+-----------------+-------------------+------------+---------------+--------------+-----+
|window_start|window_end|AIRLINE|avg_arrival_delay|avg_departure_delay|flight_count|delayed_flights|delay_rate_pct|ALERT|
+------------+----------+-------+-----------------+-------------------+------------+---------------+--------------+-----+
+------------+----------+-------+-----------------+-------------------+------------+---------------+--------------+-----+


[23:14:08] Stream status: Processing new data
  Input rows/sec:     0.0
  Processed rows/sec: 0.4
  Batch ID:           0

--- Kết quả batch hiện tại ---
+-------------------+-------------

## 8. Explain execution plan

In [12]:
print("=== EXECUTION PLAN (Structured Streaming) ===")
parsed_stream.explain(extended=True)

=== EXECUTION PLAN (Structured Streaming) ===
== Parsed Logical Plan ==
'Filter 'isNotNull('ARRIVAL_DELAY)
+- ~Filter (CANCELLED#22 = 0)
   +- ~Project [data#14.AIRLINE AS AIRLINE#16, data#14.ORIGIN_AIRPORT AS ORIGIN_AIRPORT#17, data#14.DEST_AIRPORT AS DEST_AIRPORT#18, data#14.DEPARTURE_DELAY AS DEPARTURE_DELAY#19, data#14.ARRIVAL_DELAY AS ARRIVAL_DELAY#20, data#14.DISTANCE AS DISTANCE#21, data#14.CANCELLED AS CANCELLED#22, event_time#15]
      +- ~Project [from_json(StructField(AIRLINE,StringType,true), StructField(ORIGIN_AIRPORT,StringType,true), StructField(DEST_AIRPORT,StringType,true), StructField(DEPARTURE_DELAY,FloatType,true), StructField(ARRIVAL_DELAY,FloatType,true), StructField(DISTANCE,FloatType,true), StructField(CANCELLED,IntegerType,true), cast(value#8 as string), Some(Asia/Saigon), false) AS data#14, timestamp#12 AS event_time#15]
         +- ~StreamingRelationV2 org.apache.spark.sql.kafka010.KafkaSourceProvider@4a802c6f, kafka, org.apache.spark.sql.kafka010.KafkaSource

## Save results

In [13]:
result_df = spark.sql("""
    SELECT * FROM flight_alerts
    ORDER BY avg_arrival_delay DESC
""")
result_df.show(50, truncate=False)
result_df.toPandas().to_csv("streaming_results.csv", index=False)
print("Saved to streaming_results.csv")

+-------------------+-------------------+-------+------------------+-------------------+------------+---------------+--------------+-------------------------------+
|window_start       |window_end         |AIRLINE|avg_arrival_delay |avg_departure_delay|flight_count|delayed_flights|delay_rate_pct|ALERT                          |
+-------------------+-------------------+-------+------------------+-------------------+------------+---------------+--------------+-------------------------------+
|2026-06-02 23:19:00|2026-06-02 23:20:00|F9     |82.58             |73.72              |50          |41             |82.0          |WARNING: HIGH DELAY (>15.0 min)|
|2026-06-02 23:18:00|2026-06-02 23:19:00|F9     |59.19565217391305 |46.67391304347826  |46          |30             |65.22         |WARNING: HIGH DELAY (>15.0 min)|
|2026-06-02 23:19:00|2026-06-02 23:20:00|AA     |52.34615384615385 |52.06804733727811  |338         |225            |66.57         |WARNING: HIGH DELAY (>15.0 min)|
|2026-06-0

## 9. Stop streaming & cleanup

In [14]:
if query.isActive:
    query.stop()

spark.stop()
print("Streaming stopped. SparkSession closed.")

Streaming stopped. SparkSession closed.


## Summary

| Thành phần | Chi tiết                                                                                    |
|---|---------------------------------------------------------------------------------------------|
| Source | Kafka topic `flight-stream`                                                                 |
| Format | JSON records từ `kafka_producer.py`                                                         |
| Window | Tumbling window 1 phút                                                                      |
| Aggregation | Avg delay, flight count, delay rate per carrier                                             |
| Alert | Carrier có avg arrival delay > 15 phút                                                      |
| Sink | Console output, trigger mỗi 30 giây                                                         |
| Latency vs Batch | Streaming xử lý liên tục theo micro-batch (~30s latency) thay vì đợi toàn bộ data như batch |